# 14.4 MLP activation 可以怎麼選？

# 第 14 章：逐項理解現代文字模型的設計

前置：第4章基準。這章像替熟悉的車逐一換零件，每節可獨立進入；不要同時換六項再猜哪項有效。結構／數值可先CPU驗證，品質與成本需匹配預算後訓練。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：成對feature像鐘面指針：用相對旋轉記位置**

單一pair的旋轉示意；不同pair有不同角速度。

這張圖保留靜態標註；先核對箭頭、位置與每個數字的意思。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rope.svg")))

**先想一想：**x=-1與x=2時，兩種activation誰保留負值？

activation決定特徵如何彎曲。GELU平滑地保留部分負值，ReLU²把正值平方，對大正值更敏感；比較也要控制參數與預算。

**把直覺寫成數學：**ReLU²(x)=max(0,x)²；GELU≈x·Φ(x)。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
x = torch.linspace(-2, 2, 9)
print("x", x, "GELU", F.gelu(x), "ReLU²", F.relu(x).square())

**如何讀結果：**函數形狀能先比較；模型品質在同樣FFN寬度／budget下另訓練，不替未測結果排名。

**只改一件事：**只把x範圍擴到±4。
